# 00_04: HuggingFace Preprocessors & Feature Extractors

**Learning Objectives:**
- Understand the unified preprocessing layer that bridges raw data to model-ready tensors
- Use `AutoImageProcessor` to prepare images for vision models
- Use `AutoFeatureExtractor` to prepare audio for speech models
- Use `AutoProcessor` for multimodal models that combine text + images
- Master padding, truncation, and batching strategies for text
- Visualize what preprocessing actually does to your data

**Prerequisites:** Notebooks 00_01 through 00_03. No GPU required.

## Prerequisites

| Requirement | Specification |
|-------------|---------------|
| **CPU** | Any modern CPU |
| **RAM** | 4GB minimum |
| **Storage** | ~1.5GB for model downloads |
| **GPU** | Not required |
| **Internet** | Required for first-time model downloads |

## Expected Behaviors

### What You'll See
- Tokenizers convert text to padded/truncated integer sequences with attention masks
- Image processors resize, crop, and normalize images to specific pixel value ranges
- Audio feature extractors convert waveforms to mel spectrograms
- `AutoProcessor` unifies text + image preprocessing for multimodal models like CLIP

### First Run
- Several model downloads (~1.5GB total): DistilBERT tokenizer, ViT image processor, Whisper feature extractor, CLIP processor
- All synthetic data is generated in-notebook (no external files needed)

### Common Observations
- Preprocessed images have pixel values centered around 0 (not 0-255)
- Attention masks mark which tokens are real vs padding
- Audio inputs get resampled to the model's expected sample rate (usually 16kHz)

## Overview

Every HuggingFace model expects input in a very specific format — a particular tensor shape, value range, and data type. The job of a **preprocessor** is to bridge the gap between your raw data (text strings, JPEG images, WAV audio) and the model-ready tensors.

HuggingFace provides a hierarchy of preprocessors, each specialized for a data modality:

| Raw Data | Preprocessor Class | Output | Used By |
|----------|-------------------|--------|--------|
| Text strings | `AutoTokenizer` | `input_ids`, `attention_mask` | BERT, GPT-2, T5 |
| Images (PIL/numpy) | `AutoImageProcessor` | `pixel_values` | ViT, DETR, SAM |
| Audio (waveform) | `AutoFeatureExtractor` | `input_features` | Whisper, Wav2Vec2 |
| Text + Images | `AutoProcessor` | All of the above | CLIP, BLIP, LLaVA |

The key insight: **you must always use the same preprocessor that the model was trained with**. A ViT model trained with ImageNet normalization will produce garbage if you feed it unnormalized pixels. HuggingFace handles this automatically — each model's preprocessor config is stored alongside its weights on the Hub.

## Setup and Installation

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import random
import transformers
from transformers import (
    AutoTokenizer,
    AutoImageProcessor,
    AutoFeatureExtractor,
    AutoProcessor,
    AutoModel,
)
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import sys
import time

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Part 1: Text Preprocessing Patterns

You already know that tokenizers convert text to token IDs (notebook 00_01). But in practice, text preprocessing is more nuanced — you need to handle **batches** of different-length sequences, decide on **padding** and **truncation** strategies, and understand **attention masks**. These choices directly affect model performance and memory usage.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# A batch of sentences with very different lengths
texts = [
    'Hi.',
    'This is a medium length sentence for testing.',
    'This is a much longer sentence that contains significantly more tokens and will help us understand how padding and truncation strategies affect the final tensor shapes.',
]

# Tokenize WITHOUT padding — each sequence has different length
no_pad = tokenizer(texts)
print('=== Without Padding ===')
for i, (text, ids) in enumerate(zip(texts, no_pad['input_ids'])):
    print(f'  Text {i}: {len(ids)} tokens — "{text[:40]}..."' if len(text) > 40 else f'  Text {i}: {len(ids)} tokens — "{text}"')
print(f'\nProblem: sequences have lengths {[len(ids) for ids in no_pad["input_ids"]]} — cannot stack into a tensor!')

### Padding Strategies

To batch sequences into a single tensor, they must all have the same length. **Padding** adds special `[PAD]` tokens to shorter sequences. HuggingFace offers three strategies:

| Strategy | `padding=` | Behavior |
|----------|-----------|----------|
| No padding | `False` | Sequences keep their natural length (can't batch) |
| Longest | `True` or `'longest'` | Pad to the longest sequence in the batch |
| Max length | `'max_length'` | Pad to `max_length` (or model max if not specified) |

In [ ]:
# Strategy 1: Pad to longest in batch
longest_pad = tokenizer(texts, padding='longest', return_tensors='pt')

# Strategy 2: Pad to a fixed max_length
MAX_LENGTH = 50
fixed_pad = tokenizer(texts, padding='max_length', max_length=MAX_LENGTH, truncation=True, return_tensors='pt')

print('=== Padding Strategy Comparison ===')
comparison = pd.DataFrame({
    'Strategy': ['padding="longest"', f'padding="max_length" (max_length={MAX_LENGTH})'],
    'Tensor Shape': [str(tuple(longest_pad['input_ids'].shape)), str(tuple(fixed_pad['input_ids'].shape))],
    'Pad Token Count': [
        int((longest_pad['input_ids'] == tokenizer.pad_token_id).sum()),
        int((fixed_pad['input_ids'] == tokenizer.pad_token_id).sum()),
    ],
    'Memory (elements)': [
        int(longest_pad['input_ids'].numel()),
        int(fixed_pad['input_ids'].numel()),
    ],
})
comparison

### Attention Masks: How Models Ignore Padding

Padding tokens are meaningless — the model should not attend to them. The **attention mask** is a binary tensor that tells the model which tokens are real (1) and which are padding (0). Without it, the model would treat `[PAD]` tokens as meaningful input, degrading performance.

In [ ]:
def visualize_attention_mask(
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    tokenizer: AutoTokenizer,
) -> None:
    """Visualize which tokens are real vs padding using a heatmap.

    Args:
        input_ids: Token ID tensor of shape (batch_size, seq_len).
        attention_mask: Binary mask tensor of same shape.
        tokenizer: Tokenizer for decoding token IDs.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))

    # Token IDs heatmap
    axes[0].imshow(input_ids.numpy(), aspect='auto', cmap='viridis')
    axes[0].set_title('Token IDs')
    axes[0].set_xlabel('Position')
    axes[0].set_ylabel('Sequence')

    # Attention mask heatmap
    axes[1].imshow(attention_mask.numpy(), aspect='auto', cmap='coolwarm', vmin=0, vmax=1)
    axes[1].set_title('Attention Mask (blue=padding, red=real)')
    axes[1].set_xlabel('Position')
    axes[1].set_ylabel('Sequence')

    plt.tight_layout()
    plt.show()

    # Text summary
    for i in range(input_ids.shape[0]):
        real_tokens = int(attention_mask[i].sum())
        total_tokens = attention_mask.shape[1]
        print(f'  Sequence {i}: {real_tokens}/{total_tokens} real tokens ({real_tokens/total_tokens:.0%})')


print('=== Attention Mask Visualization ===')
visualize_attention_mask(longest_pad['input_ids'], longest_pad['attention_mask'], tokenizer)

### Truncation

When sequences exceed the model's maximum length (e.g., 512 tokens for BERT), you must **truncate**. Without truncation, the model will raise an error or produce incorrect results. Always enable truncation when processing user-provided text of unknown length.

In [ ]:
# Demonstrate truncation
long_text = 'This is a word. ' * 200  # ~600 tokens

# Without truncation — will exceed model max length
no_trunc = tokenizer(long_text)
print(f'Without truncation: {len(no_trunc["input_ids"])} tokens')
print(f'Model max length: {tokenizer.model_max_length}')

# With truncation
truncated = tokenizer(long_text, truncation=True, return_tensors='pt')
print(f'With truncation: {truncated["input_ids"].shape[1]} tokens')

# With custom max_length
CUSTOM_MAX = 64
custom_trunc = tokenizer(long_text, truncation=True, max_length=CUSTOM_MAX, return_tensors='pt')
print(f'With max_length={CUSTOM_MAX}: {custom_trunc["input_ids"].shape[1]} tokens')

# Best practice: always use both padding and truncation for batches
print(f'\nBest practice pattern:')
print(f'  tokenizer(texts, padding=True, truncation=True, return_tensors="pt")')

## Part 2: Image Preprocessing

Vision models like ViT, DETR, and SAM don't process raw pixels directly. An **image processor** applies a specific sequence of transformations — resize, center crop, normalization — to convert any input image into the exact format the model was trained on. The specific transforms vary by model, which is why you must always load the processor that matches your model.

In [ ]:
# Option 1: Small Model (CPU-friendly, recommended for beginners)
IMAGE_MODEL_NAME = 'google/vit-base-patch16-224'  # 346MB

# Option 2: Large Model (GPU-optimized)
# IMAGE_MODEL_NAME = 'google/vit-large-patch16-224'  # 1.2GB

image_processor = AutoImageProcessor.from_pretrained(IMAGE_MODEL_NAME)

# Inspect the processor configuration
print('=== Image Processor Configuration ===')
config_data = {
    'Property': ['Image Size', 'Resize', 'Center Crop', 'Normalize', 'Mean (RGB)', 'Std (RGB)', 'Rescale Factor'],
    'Value': [
        str(image_processor.size),
        str(image_processor.do_resize),
        str(image_processor.do_center_crop) if hasattr(image_processor, 'do_center_crop') else 'N/A',
        str(image_processor.do_normalize),
        str([f'{v:.4f}' for v in image_processor.image_mean]),
        str([f'{v:.4f}' for v in image_processor.image_std]),
        str(image_processor.rescale_factor) if hasattr(image_processor, 'rescale_factor') else 'N/A',
    ],
}
pd.DataFrame(config_data)

In [ ]:
def create_test_image(width: int = 640, height: int = 480) -> Image.Image:
    """Create a synthetic test image with colored regions and a gradient.

    Args:
        width: Image width in pixels.
        height: Image height in pixels.

    Returns:
        A PIL RGB image with colored quadrants and a diagonal gradient.
    """
    img_array = np.zeros((height, width, 3), dtype=np.uint8)

    # Four colored quadrants
    half_h, half_w = height // 2, width // 2
    img_array[:half_h, :half_w] = [220, 50, 50]     # Red (top-left)
    img_array[:half_h, half_w:] = [50, 180, 50]     # Green (top-right)
    img_array[half_h:, :half_w] = [50, 50, 200]     # Blue (bottom-left)
    img_array[half_h:, half_w:] = [200, 200, 50]    # Yellow (bottom-right)

    # Add a diagonal gradient overlay
    for y in range(height):
        for x in range(width):
            blend = int(((x + y) / (width + height)) * 80)
            img_array[y, x] = np.clip(img_array[y, x].astype(int) + blend, 0, 255).astype(np.uint8)

    return Image.fromarray(img_array)


test_image = create_test_image(640, 480)
print(f'Original image: {test_image.size} (W×H), mode={test_image.mode}')

In [ ]:
# Preprocess the image
processed = image_processor(test_image, return_tensors='pt')
pixel_values = processed['pixel_values']

print('=== Before vs After Preprocessing ===')
before_after = pd.DataFrame({
    'Property': ['Shape', 'Dtype', 'Value Range (min)', 'Value Range (max)', 'Mean', 'Std'],
    'Original Image': [
        f'{test_image.size[1]}×{test_image.size[0]}×3 (H×W×C)',
        'uint8',
        f'{np.array(test_image).min()}',
        f'{np.array(test_image).max()}',
        f'{np.array(test_image).mean():.2f}',
        f'{np.array(test_image).std():.2f}',
    ],
    'Preprocessed Tensor': [
        f'{pixel_values.shape} (B×C×H×W)',
        str(pixel_values.dtype),
        f'{pixel_values.min():.4f}',
        f'{pixel_values.max():.4f}',
        f'{pixel_values.mean():.4f}',
        f'{pixel_values.std():.4f}',
    ],
})
before_after

In [ ]:
def visualize_preprocessing(
    original: Image.Image,
    pixel_values: torch.Tensor,
    image_mean: list[float],
    image_std: list[float],
) -> None:
    """Show original image alongside preprocessed version (denormalized for display).

    Args:
        original: Original PIL image.
        pixel_values: Preprocessed tensor of shape (1, C, H, W).
        image_mean: Per-channel mean used for normalization.
        image_std: Per-channel std used for normalization.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original
    axes[0].imshow(original)
    axes[0].set_title(f'Original ({original.size[0]}×{original.size[1]})')
    axes[0].axis('off')

    # Raw preprocessed (normalized — hard to see)
    raw_img = pixel_values[0].permute(1, 2, 0).numpy()
    axes[1].imshow(raw_img, cmap='viridis')
    axes[1].set_title('Normalized Tensor (raw values)')
    axes[1].axis('off')

    # Denormalized (undo normalization for human viewing)
    mean = np.array(image_mean)
    std = np.array(image_std)
    denorm_img = raw_img * std + mean
    denorm_img = np.clip(denorm_img, 0, 1)
    axes[2].imshow(denorm_img)
    axes[2].set_title(f'Denormalized ({raw_img.shape[0]}×{raw_img.shape[1]})')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()


visualize_preprocessing(
    test_image,
    pixel_values,
    image_processor.image_mean,
    image_processor.image_std,
)

Notice three things: (1) the image was **resized** from 640×480 to 224×224, (2) pixel values shifted from 0-255 integers to roughly -2 to +2 floats centered around 0, and (3) the channel order changed from HWC (height, width, channels) to CHW (channels, height, width) — the format PyTorch models expect.

In [ ]:
# Batch preprocessing — process multiple images at once
images = [
    create_test_image(640, 480),
    create_test_image(320, 320),
    create_test_image(800, 600),
]

batch_processed = image_processor(images, return_tensors='pt')

print('=== Batch Image Preprocessing ===')
print(f'Input: {len(images)} images of sizes {[img.size for img in images]}')
print(f'Output: pixel_values shape = {batch_processed["pixel_values"].shape}')
print(f'All images resized to the same dimensions — ready for batched inference!')

## Part 3: Audio Preprocessing

Audio models like Whisper and Wav2Vec2 don't process raw waveforms directly. An **audio feature extractor** converts a 1D waveform into a 2D **mel spectrogram** — a time-frequency representation that captures the information models need. The feature extractor also handles resampling (converting between sample rates) and padding to fixed lengths.

In [ ]:
# Option 1: Small Model (CPU-friendly, recommended for beginners)
AUDIO_MODEL_NAME = 'openai/whisper-tiny'  # 72MB

# Option 2: Large Model (GPU-optimized)
# AUDIO_MODEL_NAME = 'openai/whisper-small'  # 461MB

feature_extractor = AutoFeatureExtractor.from_pretrained(AUDIO_MODEL_NAME)

# Inspect configuration
print('=== Audio Feature Extractor Configuration ===')
audio_config = pd.DataFrame({
    'Property': [
        'Expected Sample Rate',
        'Feature Size',
        'Chunk Length (seconds)',
        'Number of Mel Bins',
        'Padding Value',
        'Return Attention Mask',
    ],
    'Value': [
        f'{feature_extractor.sampling_rate} Hz',
        str(feature_extractor.feature_size),
        str(getattr(feature_extractor, 'chunk_length', 'N/A')),
        str(getattr(feature_extractor, 'nb_max_frames', getattr(feature_extractor, 'feature_size', 'N/A'))),
        str(feature_extractor.padding_value),
        str(feature_extractor.return_attention_mask),
    ],
})
audio_config

In [ ]:
def create_synthetic_audio(
    duration: float = 3.0,
    sample_rate: int = 16000,
    frequencies: list[float] | None = None,
) -> np.ndarray:
    """Generate a synthetic audio signal by combining sine waves.

    Args:
        duration: Duration in seconds.
        sample_rate: Samples per second.
        frequencies: List of frequencies (Hz) to combine. Defaults to a chord.

    Returns:
        1D numpy array of float32 audio samples in [-1, 1].
    """
    if frequencies is None:
        frequencies = [261.63, 329.63, 392.00]  # C major chord (C4, E4, G4)

    t = np.linspace(0, duration, int(sample_rate * duration), dtype=np.float32)
    signal = np.zeros_like(t)

    for freq in frequencies:
        signal += np.sin(2 * np.pi * freq * t)

    # Normalize to [-1, 1]
    signal = signal / np.max(np.abs(signal))

    # Add slight fade in/out to avoid clicks
    fade_samples = int(0.05 * sample_rate)
    signal[:fade_samples] *= np.linspace(0, 1, fade_samples)
    signal[-fade_samples:] *= np.linspace(1, 0, fade_samples)

    return signal


SAMPLE_RATE = 16000
DURATION = 3.0
audio_signal = create_synthetic_audio(DURATION, SAMPLE_RATE)
print(f'Synthetic audio: {len(audio_signal)} samples, {DURATION}s at {SAMPLE_RATE}Hz')
print(f'Value range: [{audio_signal.min():.4f}, {audio_signal.max():.4f}]')

In [ ]:
# Process the audio
audio_features = feature_extractor(
    audio_signal,
    sampling_rate=SAMPLE_RATE,
    return_tensors='pt',
)

print('=== Before vs After Audio Preprocessing ===')
audio_comparison = pd.DataFrame({
    'Property': ['Shape', 'Dtype', 'Value Range (min)', 'Value Range (max)'],
    'Raw Waveform': [
        f'({len(audio_signal)},) — 1D signal',
        str(audio_signal.dtype),
        f'{audio_signal.min():.4f}',
        f'{audio_signal.max():.4f}',
    ],
    'Mel Spectrogram': [
        f'{audio_features["input_features"].shape} — (batch, mel_bins, time_frames)',
        str(audio_features['input_features'].dtype),
        f'{audio_features["input_features"].min():.4f}',
        f'{audio_features["input_features"].max():.4f}',
    ],
})
audio_comparison

In [ ]:
def visualize_audio_preprocessing(
    waveform: np.ndarray,
    spectrogram: torch.Tensor,
    sample_rate: int,
) -> None:
    """Plot the raw waveform and its mel spectrogram side by side.

    Args:
        waveform: 1D numpy array of audio samples.
        spectrogram: Mel spectrogram tensor of shape (1, mel_bins, time_frames).
        sample_rate: Audio sample rate in Hz.
    """
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))

    # Waveform
    time_axis = np.arange(len(waveform)) / sample_rate
    axes[0].plot(time_axis, waveform, linewidth=0.5, color='steelblue')
    axes[0].set_title('Raw Waveform')
    axes[0].set_xlabel('Time (seconds)')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_xlim(0, time_axis[-1])

    # Mel spectrogram
    spec_data = spectrogram[0].numpy()
    axes[1].imshow(spec_data, aspect='auto', origin='lower', cmap='viridis')
    axes[1].set_title('Mel Spectrogram (model input)')
    axes[1].set_xlabel('Time Frames')
    axes[1].set_ylabel('Mel Frequency Bins')

    plt.tight_layout()
    plt.show()


visualize_audio_preprocessing(audio_signal, audio_features['input_features'], SAMPLE_RATE)

The mel spectrogram transforms a 1D time-domain signal into a 2D time-frequency representation. The y-axis shows frequency bands (mel-scaled to match human hearing), and the x-axis shows time frames. Brighter regions indicate more energy at that frequency and time. This 2D representation is what audio models actually "see" — it's analogous to how vision models see pixel grids.

In [ ]:
# What happens with different audio lengths?
short_audio = create_synthetic_audio(duration=1.0, sample_rate=SAMPLE_RATE)
long_audio = create_synthetic_audio(duration=10.0, sample_rate=SAMPLE_RATE)

short_features = feature_extractor(short_audio, sampling_rate=SAMPLE_RATE, return_tensors='pt')
long_features = feature_extractor(long_audio, sampling_rate=SAMPLE_RATE, return_tensors='pt')

print('=== Audio Length Handling ===')
length_df = pd.DataFrame({
    'Input Duration': ['1.0s', '3.0s', '10.0s'],
    'Input Samples': [f'{len(short_audio):,}', f'{len(audio_signal):,}', f'{len(long_audio):,}'],
    'Output Shape': [
        str(tuple(short_features['input_features'].shape)),
        str(tuple(audio_features['input_features'].shape)),
        str(tuple(long_features['input_features'].shape)),
    ],
})
length_df

Notice that Whisper pads all audio to a fixed 30-second window (3000 time frames) regardless of input length. Shorter clips get zero-padded, while longer clips would be chunked. This fixed-length approach simplifies batching but uses more memory for short clips.

## Part 4: The Unified AutoProcessor

Multimodal models like CLIP, BLIP, and LLaVA process **both text and images** (or other modality combinations). Instead of managing separate tokenizers and image processors, HuggingFace provides `AutoProcessor` — a unified wrapper that handles all modalities through a single interface.

`AutoProcessor` automatically detects what preprocessing components the model needs and wraps them together. You can still access the individual components via `.tokenizer` and `.image_processor` attributes.

In [ ]:
# Option 1: Small Model (CPU-friendly, recommended for beginners)
MULTIMODAL_MODEL_NAME = 'openai/clip-vit-base-patch32'  # 605MB

# Option 2: Large Model (GPU-optimized)
# MULTIMODAL_MODEL_NAME = 'openai/clip-vit-large-patch14'  # 1.7GB

processor = AutoProcessor.from_pretrained(MULTIMODAL_MODEL_NAME)

print('=== AutoProcessor Components ===')
print(f'Processor type: {type(processor).__name__}')
print(f'Tokenizer: {type(processor.tokenizer).__name__}')
print(f'Image Processor: {type(processor.image_processor).__name__}')
print(f'\nThe processor wraps both a tokenizer and an image processor into one object.')

In [ ]:
# Process text + image together
test_texts = ['a photo of a cat', 'a photo of a dog', 'a red and blue pattern']
test_image = create_test_image(400, 300)

# Single call handles both modalities
inputs = processor(
    text=test_texts,
    images=test_image,
    return_tensors='pt',
    padding=True,
)

print('=== AutoProcessor Output ===')
for key, tensor in inputs.items():
    print(f'  {key}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}')

print(f'\nText was tokenized into input_ids + attention_mask')
print(f'Image was preprocessed into pixel_values')
print(f'Both ready for the model in a single dictionary!')

In [ ]:
def compare_separate_vs_unified(
    processor: AutoProcessor,
    text: str,
    image: Image.Image,
) -> pd.DataFrame:
    """Compare separate preprocessing vs unified AutoProcessor.

    Args:
        processor: An AutoProcessor instance.
        text: Input text.
        image: Input PIL image.

    Returns:
        DataFrame comparing the two approaches.
    """
    # Approach 1: Separate preprocessors
    text_inputs = processor.tokenizer(text, return_tensors='pt', padding=True)
    image_inputs = processor.image_processor(image, return_tensors='pt')

    # Approach 2: Unified processor
    unified_inputs = processor(text=text, images=image, return_tensors='pt', padding=True)

    # Verify they produce identical results
    text_match = torch.equal(text_inputs['input_ids'], unified_inputs['input_ids'])
    image_match = torch.equal(image_inputs['pixel_values'], unified_inputs['pixel_values'])

    return pd.DataFrame({
        'Aspect': ['Lines of code', 'Text output matches?', 'Image output matches?', 'Recommended?'],
        'Separate (tokenizer + image_processor)': ['2 calls', str(text_match), str(image_match), 'Only when you need modality-specific control'],
        'Unified (AutoProcessor)': ['1 call', str(text_match), str(image_match), 'Yes — simpler and less error-prone'],
    })


compare_separate_vs_unified(processor, 'a colorful test image', test_image)

The unified `AutoProcessor` produces identical output to using the components separately — it's pure convenience. Use it as your default, and only drop down to individual preprocessors when you need modality-specific control (e.g., different padding strategies for text vs images).

## Part 5: Preprocessing Best Practices

Preprocessing errors are a common source of bugs in ML pipelines. Here are the patterns and pitfalls to watch for.

In [ ]:
def preprocess_text_batch(
    texts: list[str],
    tokenizer: AutoTokenizer,
    max_length: int = 128,
) -> dict[str, torch.Tensor]:
    """Robust text preprocessing with padding, truncation, and validation.

    Args:
        texts: List of input strings.
        tokenizer: HuggingFace tokenizer.
        max_length: Maximum sequence length.

    Returns:
        Dictionary with input_ids and attention_mask tensors.
    """
    # Best practice: always use padding + truncation + return_tensors
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )

    return encoded


# Demonstrate the pattern
sample_texts = [
    'Short text.',
    'A slightly longer piece of text to process.',
    'An even longer text that goes on and on ' * 10,  # Will be truncated
]

batch = preprocess_text_batch(sample_texts, tokenizer, max_length=32)
print('=== Robust Batch Preprocessing ===')
print(f'Input: {len(sample_texts)} texts of varying length')
print(f'Output shape: {tuple(batch["input_ids"].shape)}')
print(f'All sequences: padded to same length, truncated if needed, ready for model')

for i in range(len(sample_texts)):
    real_tokens = int(batch['attention_mask'][i].sum())
    total_tokens = batch['attention_mask'].shape[1]
    status = 'truncated' if real_tokens == 32 else 'padded'
    print(f'  Text {i}: {real_tokens}/{total_tokens} real tokens ({status})')

In [ ]:
# Common pitfall: mismatched preprocessor and model
print('=== Preprocessor-Model Matching Rules ===')
rules = pd.DataFrame({
    'Rule': [
        'Always load preprocessor from the same model name',
        'Never manually normalize images — let the processor do it',
        'Always resample audio to the model\'s expected sample rate',
        'Use return_tensors="pt" for PyTorch models',
        'Always enable truncation for user-provided text',
    ],
    'Why': [
        'Different models use different normalization, vocab, and formats',
        'Wrong mean/std values produce garbage predictions',
        'Wrong sample rate shifts all frequencies — model hears noise',
        'Returns stacked tensors instead of Python lists',
        'Prevents index-out-of-range errors on long inputs',
    ],
    'Pattern': [
        'AutoXxx.from_pretrained(MODEL_NAME)',
        'processor(image, return_tensors="pt")',
        'feature_extractor(audio, sampling_rate=16000)',
        'tokenizer(text, return_tensors="pt")',
        'tokenizer(text, truncation=True)',
    ],
})
rules

## Performance Benchmarking

Let's measure preprocessing speed across modalities and compare fast vs slow tokenizers.

In [ ]:
def benchmark_text_preprocessing(
    texts: list[str],
    tokenizer: AutoTokenizer,
    num_runs: int = 20,
) -> dict[str, float]:
    """Benchmark tokenizer speed for individual vs batch processing.

    Args:
        texts: List of input texts.
        tokenizer: HuggingFace tokenizer.
        num_runs: Number of timing runs.

    Returns:
        Dictionary with timing results in milliseconds.
    """
    # Individual processing
    individual_times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        for text in texts:
            tokenizer(text, padding='max_length', max_length=128, truncation=True, return_tensors='pt')
        individual_times.append(time.perf_counter() - start)

    # Batch processing
    batch_times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        tokenizer(texts, padding='max_length', max_length=128, truncation=True, return_tensors='pt')
        batch_times.append(time.perf_counter() - start)

    return {
        'individual_ms': np.mean(individual_times) * 1000,
        'batch_ms': np.mean(batch_times) * 1000,
    }


benchmark_texts = [
    f'This is test sentence number {i} for benchmarking tokenizer performance.'
    for i in range(50)
]

results = benchmark_text_preprocessing(benchmark_texts, tokenizer)

print('=== Preprocessing Speed (50 texts) ===')
speed_df = pd.DataFrame({
    'Approach': ['Individual (loop)', 'Batch (single call)'],
    'Time (ms)': [f'{results["individual_ms"]:.1f}', f'{results["batch_ms"]:.1f}'],
    'Speedup': ['1.0x (baseline)', f'{results["individual_ms"] / results["batch_ms"]:.1f}x'],
})
speed_df

In [ ]:
# Cross-modality preprocessing summary
print('=== Preprocessor Summary ===')
summary = pd.DataFrame({
    'Modality': ['Text', 'Image', 'Audio', 'Multimodal'],
    'AutoClass': ['AutoTokenizer', 'AutoImageProcessor', 'AutoFeatureExtractor', 'AutoProcessor'],
    'Input Format': ['str or list[str]', 'PIL Image or numpy', '1D numpy array', 'text + image(s)'],
    'Key Output': ['input_ids, attention_mask', 'pixel_values', 'input_features', 'All combined'],
    'Key Transform': ['Tokenize + pad + truncate', 'Resize + normalize', 'Resample + mel spectrogram', 'Wraps text + image'],
})
summary

## Exercises

1. **Padding experiment**: Tokenize 10 sentences of wildly different lengths with `padding='longest'` vs `padding='max_length'` (max_length=512). Compare the total number of padding tokens. When does each strategy waste less memory?

2. **Image processor comparison**: Load image processors for `google/vit-base-patch16-224` and `facebook/detr-resnet-50`. Compare their configurations (image size, normalization values). Why might they differ?

3. **Audio resampling**: Create synthetic audio at 44100Hz (CD quality) and process it with Whisper's feature extractor (which expects 16000Hz). Does the feature extractor handle the mismatch, or do you need to resample manually?

4. **Custom CLIP experiment**: Use the CLIP `AutoProcessor` to process 5 different text descriptions against 3 different synthetic images. Inspect the output shapes and verify that text and image batch dimensions are independent.

## Key Takeaways

- **Every model has a matching preprocessor** — always load it from the same model name to ensure compatible normalization, vocabulary, and formatting
- **Text preprocessing** involves tokenization, padding (to equalize lengths), truncation (to respect limits), and attention masks (to ignore padding)
- **Image preprocessing** resizes, crops, and normalizes pixel values from 0-255 integers to model-specific float ranges centered near zero
- **Audio preprocessing** converts 1D waveforms to 2D mel spectrograms — the time-frequency representation that speech models actually process
- **`AutoProcessor`** unifies text + image (or other modality) preprocessing into a single call — use it as your default for multimodal models

## Next Steps & Resources

**Next notebook**: [00_05 — Model Configuration & Customization](00_05_model_configuration_customization.ipynb) — learn how to inspect, modify, and build model architectures through config objects.

**Resources:**
- [Preprocessing Tutorial](https://huggingface.co/docs/transformers/preprocessing) — Official preprocessing guide
- [AutoProcessor API](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoProcessor) — AutoProcessor reference
- [Image Processor API](https://huggingface.co/docs/transformers/main_classes/image_processor) — Image preprocessing reference
- [Feature Extractor API](https://huggingface.co/docs/transformers/main_classes/feature_extractor) — Audio preprocessing reference
- [Padding & Truncation](https://huggingface.co/docs/transformers/pad_truncation) — Detailed padding strategies guide